If you want to type along with me, head to [this notebook](https://humboldt.cloudbank.2i2c.cloud/hub/user-redirect/git-pull?repo=https%3A%2F%2Fgithub.com%2Fmlekimchi%2Fdata111_fall26&branch=main&urlpath=tree%2Fdata111_fall26%2Flectures%2Flec20_live.ipynb) instead. If you prefer follow along by executing the cells, stay in this notebook.

# Lecture 20: A/B Testing and Randomized Experiments
v.ekc

The permutation test, end to end — then the payoff: with **random assignment**, rejecting the null supports a *causal* conclusion. (Slides: KL Lecture 20 deck.)

**Today's sections:**
1. Comparing two samples
2. Random permutation
3. Simulation under the null
4. The permutation test
5. Try It — a randomized controlled experiment

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
## 1. Comparing Two Samples

Smokers vs. non-smokers, one more time — this run is the reference version. (KL deck, slides 10–15.)

In [ ]:
births = Table.read_table('baby.csv')
births

In [ ]:
smoking_and_birthweight = births.select('Maternal Smoker', 'Birth Weight')

In [ ]:
smoking_and_birthweight.group('Maternal Smoker')

In [ ]:
smoking_and_birthweight.hist('Birth Weight', group='Maternal Smoker')

# Test Statistic

**Question:** What values of our statistic are in favor of the alternative: positive or negative?

In [ ]:
means_table = smoking_and_birthweight.group('Maternal Smoker', np.average)
means_table

In [ ]:
means = means_table.column(1)
observed_difference = means.item(1) - means.item(0)
observed_difference

In [ ]:
def difference_of_means(table, label, group_label):
    """Takes: name of table, column label of numerical variable,
    column label of group-label variable
    Returns: Difference of means of the two groups"""
    
    #table with the two relevant columns
    reduced = table.select(label, group_label)  
    
    # table containing group means
    means_table = reduced.group(group_label, np.average)
    # array of group means
    means = means_table.column(1)
    
    return means.item(1) - means.item(0)

In [ ]:
difference_of_means(births, 'Birth Weight', 'Maternal Smoker')

---
## 2. Random Permutation (Shuffling)

Under the null the labels are meaningless — shuffling them should change nothing.

In [ ]:
letters = Table().with_column('Letter', make_array('a', 'b', 'c', 'd', 'e'))

In [ ]:
letters.sample()

In [ ]:
letters.sample(with_replacement = False)

In [ ]:
letters.with_column('Shuffled', letters.sample(with_replacement = False).column(0))

---
## 3. Simulation Under the Null

In [ ]:
smoking_and_birthweight

In [ ]:
shuffled_labels = smoking_and_birthweight.sample(with_replacement=False).column('Maternal Smoker')
shuffled_labels

In [ ]:
original_and_shuffled = smoking_and_birthweight.with_column(
    'Shuffled Label', shuffled_labels
)
original_and_shuffled

In [ ]:
difference_of_means(original_and_shuffled, 'Birth Weight', 'Shuffled Label')

In [ ]:
observed_statistic = difference_of_means(original_and_shuffled, 'Birth Weight', 'Maternal Smoker')
observed_statistic

---
## 4. The Permutation Test

In [ ]:
def one_simulated_difference(table, label, group_label):
    """Takes: name of table, column label of numerical variable,
    column label of group-label variable
    Returns: Difference of means of the two groups after shuffling labels"""
    
    # array of shuffled labels
    shuffled_labels = table.sample(with_replacement = False).column(group_label)
    
    # table of numerical variable and shuffled labels
    shuffled_table = table.select(label).with_column('Shuffled Label', shuffled_labels)
    
    return difference_of_means(shuffled_table, label, 'Shuffled Label')   

In [ ]:
one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')

In [ ]:
differences = make_array()

for i in np.arange(2500):
    new_difference = one_simulated_difference(births, 'Birth Weight', 'Maternal Smoker')
    differences = np.append(differences, new_difference)

In [ ]:
Table().with_column('Difference Between Group Means', differences).hist()
print('Observed Difference:', observed_difference)
plots.title('Prediction Under the Null Hypothesis');
plots.scatter(observed_statistic, -0.01, color='red', s=120);

In [ ]:
# p-value

sum(differences <= observed_statistic)/len(differences)

> **p-value = 0.** Not one of 2500 shuffles produced a difference as extreme as the real one. But note: mothers *chose* whether to smoke — so this shows a real difference, not yet a cause. For causation we need the next section.

---
## 5. Try It: A Randomized Controlled Experiment 💉

31 chronic back-pain patients were **randomly assigned**: 15 to botulinum toxin A, 16 to saline placebo. `Result` = 1 means pain relief. (KL deck, slides 23–24.)

In [ ]:
botox = Table.read_table('bta.csv')
botox.show()

1. In one line of code: how many people in each group experienced pain relief (and how many didn't)?

2. In one line of code: what *proportion* of each group experienced relief?

In [ ]:
# Fill in — replace the ... 😅
...

In [ ]:
# Fill in — replace the ... 😅
...

<details>
<summary>💡 Possible answers — click to peek</summary>
<br>

*Counts want a pivot; proportions of a 0/1 column are just averages.*

```python
# 1.
botox.pivot('Result', 'Group')

# 2.
botox.group('Group', np.average)     # Control: 0.125, Treatment: 0.6
```

</details>

> **Why random assignment matters:** because a coin — not the patients — decided who got botox, the two groups differ *only* by treatment and chance. So when a permutation test rejects "chance," what's left is the treatment: a **causal** conclusion. That's the entire logic of clinical trials, and of Homework 7's final question.

---
> **Today's takeaway:** shuffle to simulate the null, compare to the observed gap — and if assignment was random, a rejected null means the treatment *works*.

## Appendix — Quick Reference

Full `datascience` docs: [data8.org/datascience](https://data8.org/datascience/)

### Minimal template — A/B test decision
```python
p_value = sum(differences <= observed_statistic) / len(differences)
# random assignment + tiny p-value → evidence of causation
```